# Cross-Region Inference (CRIS) Benchmark & Resilient Routing

**References:**
- Slide 29 (*Hands-On RAG with AWS*)
- Slide 46 (*Building Agentic Workflows with RAG on AWS Bedrock*)

### Why Cross-Region Inference (CRIS)?
In enterprise RAG and agentic workflows, high concurrency can trigger `ThrottlingException` (HTTP 429) on single-region endpoints. AWS Bedrock **Cross-Region Inference (CRIS)** dynamically routes requests across multiple AWS regions under the hood:
- Automatically provisions higher throughput limits.
- Absorbs regional traffic spikes and maintains lower tail latencies (p90/p99).
- Provides seamless geographic failover profiles (e.g. `us.anthropic.claude-*`, `eu.anthropic.claude-*`).

In this notebook, we benchmark Single-Region Direct endpoints vs. CRIS profiles and demonstrate a resilient failover router.

In [ ]:
from cris_benchmark_and_routing import CRISRouter, CRISBenchmarkRunner

# Initialize benchmark runner (auto-detects credentials or falls back to simulation mode)
runner = CRISBenchmarkRunner()
print(f"Benchmark Runner initialized. Mock Mode: {runner.mock_mode}")

## 1. Run Latency & Throughput Benchmark
We compare latency distributions (p50, p90, p99) and token throughput across Single-Region and CRIS US profiles.

In [ ]:
benchmark_results = runner.run_benchmark(num_requests=10, concurrency=2)

print("\n--- Benchmark Results Summary ---")
for name in ["Single-Region Direct", "CRIS US Profile"]:
    m = benchmark_results[name]
    print(f"{name}:")
    print(f"  - p50 Latency: {m['p50_latency_ms']} ms")
    print(f"  - p90 Latency: {m['p90_latency_ms']} ms")
    print(f"  - Avg Throughput: {m['avg_throughput_tok_sec']} tok/sec")

## 2. Dynamic Failover Routing Test
We simulate a 429 ThrottlingException on the primary profile to observe how the `CRISRouter` recovers transparently by rerouting to a fallback endpoint.

In [ ]:
failover_data = benchmark_results["Dynamic Failover Test"]
print("Dynamic Failover Telemetry:")
print(f"  - Primary Endpoint (Encountered Throttle): {failover_data['primary_profile']}")
print(f"  - Fallback Endpoint Used: {failover_data['failover_profile']}")
print(f"  - Total Attempts: {failover_data['total_attempts']}")
print(f"  - Success Status: {failover_data['recovered_successfully']}")
print(f"  - End-to-End Recovery Latency: {failover_data['total_latency_ms']} ms")

## 3. Architecture Best Practices
1. **Always use CRIS Model IDs (`us.` or `eu.`) in production** to aggregate service quotas across regions.
2. **Implement cross-profile fallback in clients** to withstand multi-AZ or intra-region capacity constraints without application downtime.